This notebook attempts to solve the following 2D pde with IC and BC:

\begin{equation}
\partial_t\rho = -(\nabla\cdot\mathbf{v}\rho) 
\end{equation}

\begin{equation}
\left\{ \begin{aligned}
    \rho(\mathbf{x},t) &= 1 \qquad \mathbf{x}\in(\delta\Omega_1\cup\delta\Omega_2)\\ 
    \rho(\mathbf{x},0) &= 0 \qquad \forall\mathbf{x}
\end{aligned}\right.
\end{equation}

where we assume $\mathbf{v}=(v_x, v_y)^T$.

After multiplying by a test function $q$ and integrating, we get:

$$
\iint(\partial_t\rho)qdx = -\iint (\nabla \cdot \mathbf{v}\rho)q\,dx
$$
We discretize $\rho$ into time steps $n$ and spacial steps $j$:

$$
\rho^n = \sum_i \rho^{i,n}\psi^i
$$

and now if we change test function $q$ with $\psi^j$ we can substitute and use an implicit time scheme to get the full discrete form:

$$
\sum_i\left[
    \int\psi^i\psi^jdx dx + \Delta t \int (\mathbf{v}\cdot\nabla\psi^i)\psi^jdx
\right]\rho^{i,n+1} = \sum_i\int\psi^i\psi^jdx\rho^{i,n}
$$

which we can rewrite as:

$$
(M+\Delta t C) x^{n+1} = Mx^n \\
Ax^{n+1}=Mx^n
$$

with
$$M_{ij}=\int \psi^i\psi^j dx,\\
C_{ij}=\int (\nabla\cdot\mathbf{v}\psi^i)\psi^j \,dx,\\
A_{ij}=M_{ij}+\Delta t K_{ij}$$

where we still need to compensate for the top and right boundaries for $K$.

In [46]:
using Ferrite, SparseArrays, WriteVTK

In [47]:
# Stop time, time step and velocity
dt = .005;
T = 1.3;
vx = 0.8;
vy = 1.0;
velocity = Vec((vx, vy));

name = "constant_velocity_2d_2_noibp";

In [ ]:
# generates 100 gridpoints between 
grid = generate_grid(Quadrilateral, (100, 100), Vec((0.0, 0.0)), Vec((1.0, 1.0)));

# generate the same 
ip = Lagrange{RefQuadrilateral, 1}();
qr = QuadratureRule{RefQuadrilateral}(2);
fr = FacetQuadratureRule{RefQuadrilateral};
cellvalues = CellValues(qr, ip);

# for the right and top boundary rest terms of our integration by parts
fqr = FacetQuadratureRule{RefQuadrilateral}(1);
facetvalues = FacetValues(Float64, fqr, ip);
right = getfacetset(grid, "right");
top = getfacetset(grid, "top");

In [49]:
# degree of freedom handler, 
# our finished vector is (rho^1, rho^2, ... , rho^N)
dh = DofHandler(grid)
add!(dh, :rho, ip)
close!(dh);

In [50]:
# constraint handler
ch = ConstraintHandler(dh)
r_left = getfacetset(grid, "left")
r_bottom = getfacetset(grid, "bottom")

dbc_left = Dirichlet(
    :rho,
    r_left,
    (x, t) -> 1.0
)

dbc_bottom = Dirichlet(
    :rho,
    r_bottom,
    (x, t) -> 1.0
)

add!(ch, dbc_left)
add!(ch, dbc_bottom)
close!(ch)
update!(ch, 0.0)

$$M_{ij}=\int \psi^i\psi^j dx$$

$$C_{ij}=\int (\mathbf{v}\cdot\nabla\psi^i)\psi^j \,dx$$

In [51]:
function doassemble_C!(
    C::SparseMatrixCSC, 
    cellvalues::CellValues, 
    dh::DofHandler,
    facetvalues::FacetValues,
    grid::Grid
    )

    n_basefuncs = getnbasefunctions(cellvalues)
    Ce = zeros(n_basefuncs, n_basefuncs)

    assembler = start_assemble(C)

    for cell in CellIterator(dh)

        fill!(Ce, 0)
        reinit!(cellvalues, cell)

        for q_point in 1:getnquadpoints(cellvalues)
            dx = getdetJdV(cellvalues, q_point)

            for i in 1:n_basefuncs
                psi_i = shape_value(cellvalues, q_point, i)

                for j in 1:n_basefuncs
                    dpsi_j = shape_gradient(cellvalues, q_point, j)
                    # -v \psi^i(\partial_x\psi^j)dx
                    Ce[i, j] += (velocity ⋅ dpsi_j) *  psi_i * dx
                end
            end
        end

        assemble!(assembler, celldofs(cell), Ce)
    end

    return C
end


doassemble_C! (generic function with 1 method)

In [52]:
function doassemble_M!(M::SparseMatrixCSC, cellvalues::CellValues, dh::DofHandler)

    # Allocate the element stiffness matrix
    n_basefuncs = getnbasefunctions(cellvalues)
    Me = zeros(n_basefuncs, n_basefuncs)

    # create an assembler
    assembler = start_assemble(M)

    for cell in CellIterator(dh)

        # per cell we take the local assembly matrix Me as zero
        fill!(Me, 0)

        # cellvalues contains tons and now it knows which cell we are looking at
        reinit!(cellvalues, cell)

        for q_point in 1:getnquadpoints(cellvalues)

            # not really dx but quadrature weight thingies
            dx = getdetJdV(cellvalues, q_point)

            for i in 1:n_basefuncs
                psi_i = shape_value(cellvalues, q_point, i)
                for j in 1:n_basefuncs
                    psi_j = shape_value(cellvalues, q_point, j)
                    Me[i, j] += (psi_i * psi_j) * dx
                end
            end
        end

        assemble!(assembler, celldofs(cell), Me)
    end
    return M
end

doassemble_M! (generic function with 1 method)

In [53]:
C = allocate_matrix(dh);
M = allocate_matrix(dh);

C = doassemble_C!(C, cellvalues, dh, facetvalues, grid)
M = doassemble_M!(M, cellvalues, dh)
A = (dt .* C) + M;

In [54]:
rho_n = zeros(ndofs(dh));
apply!(rho_n, ch)

rhsdata = get_rhs_data(ch, A);
apply!(A, ch);


In [55]:
pvd = paraview_collection(joinpath(name, name))

VTKGridFile(joinpath(name, name), dh) do vtk
    write_solution(vtk, dh, rho_n)
    pvd[0.0] = vtk
end

VTKGridFile for the closed file "constant_velocity_2d_2_noibp\constant_velocity_2d_2_noibp.vtu".

In [56]:
# finally the timeloop
for (step, t) in enumerate(dt:dt:T)
    #First of all, we need to update the Dirichlet boundary condition values.
    # does not really do anything since the boundary condition is independent of time?
    update!(ch, t)

    #Secondly, we compute the right-hand-side of the problem.
    b = M * rho_n
    #Then, we can apply the boundary conditions of the current time step.
    apply_rhs!(rhsdata, b, ch)

    #Finally, we can solve the time step and save the solution afterwards.
    rho = A \ b
    apply!(rho, ch)   # enforce rho(0,t)=1 exactly ??

    VTKGridFile(joinpath(name, "$(name)_1-$step"), dh) do vtk
        write_solution(vtk, dh, rho)
        pvd[t] = vtk
    end
    #At the end of the time loop, we set the previous solution to the current one and go to the next time step.
    rho_n .= rho
end

In [57]:
vtk_save(pvd);